# Phase 2 — Graph Attention Network Training

**Mental Health Monitoring System — symptom-level explainability layer**

This notebook trains the GAT that sits on top of the frozen Phase-1 backbone. It is designed
to run on Kaggle, where the Phase-1 checkpoint (`mentalroberta_phase1_final`) already lives in
`/kaggle/working/` (or as a Kaggle Dataset you've attached to this notebook).

## What changed since the original Phase-2 plan

The original plan assumed PRIMATE's 2024 reannotation (`PRIMATE.json`) covered all 9 PHQ-9
symptoms with word-level evidence spans, and that we'd train the GAT on **node-level** span
prediction. After actually inspecting the two files we have, that assumption doesn't hold:

| File | What it actually is |
|---|---|
| `primate_dataset.json` | **2,003** posts, each labelled **yes/no across all 9 PHQ-9 symptoms**. This is PRIMATE 2022 — the base dataset. |
| `PRIMATE.json` | **167** entries, but only for **one symptom — anhedonia** (loss of interest/pleasure), and only **41 of those 167** have a real character-span quote. The rest are empty (symptom wasn't mentioned/answerable). |

41 word-level spans is nowhere near enough to *train* a node-level attention objective. So the
revised design — the one this notebook implements — splits the two files into two different
jobs:

1. **Training signal (all 2,003 posts):** graph-level, 9-way multi-label classification. The GAT
   pools token-level node embeddings into one graph embedding per post, and predicts all 9 PHQ-9
   symptoms from it. This is where the model actually learns.
2. **Explainability validation (the 41 spans only):** *after* training, we run those 41
   anhedonia-annotated posts back through the trained model, pull out the GAT's attention
   weights, and check whether attention concentrates on the words a human annotator marked as
   evidence for anhedonia. This is a qualitative sanity check on explainability — it is never
   used as a training signal, and it never touches gradients.

## Why freeze the backbone here

Mental-RoBERTa was already fine-tuned in Phase 1 on Dreaddit and GoEmotions — it produces
embeddings that are already sensitive to clinical and emotional language. If we let the
transformer keep training here, its gradients (hundreds of millions of parameters) would
overwhelm the GAT's (a few thousand parameters) and the GAT would never learn anything useful.
So the backbone is a **fixed feature extractor** for the whole of Phase 2. Only the GAT trains.


## 1. Setup

In [1]:
# Run once per Kaggle session. Safe to re-run — pip skips what's already installed.
!pip install -q torch_geometric
!pip install -q -U spacy
!python -m spacy download en_core_web_sm -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 84.8 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import os
import re
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, global_mean_pool

import spacy
from transformers import AutoModel, AutoTokenizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


Device: cuda


## 2. Config — paths

In [11]:
class CFG:
    # Where the two PRIMATE files live. Adjust to wherever you've attached them
    # as Kaggle input, e.g. "/kaggle/input/primate-data/primate_dataset.json".
    PRIMATE_BASE_PATH = "/kaggle/input/datasets/shashwatkashyap12221/primate-mental-health/primate_dataset.json"
    PRIMATE_2024_PATH = "/kaggle/input/datasets/shashwatkashyap12221/primate-mental-health/PRIMATE.json"

    # Phase-1 checkpoint. This must be the folder saved by
    # AutoModel.save_pretrained() at the end of Phase 1.
    PHASE1_CHECKPOINT = "/kaggle/input/notebooks/shashwatkashyap12221/phase-1/mentalroberta_phase1_final"

    # Fallback base model, used only if the Phase-1 checkpoint isn't found —
    # lets you smoke-test this notebook's plumbing before Phase 1 is available.
    FALLBACK_BASE_MODEL = "mental/mental-roberta-base"

    MAX_TOKENS = 512          # RoBERTa hard limit
    HIDDEN_DIM = 256
    GAT_HEADS_L1 = 4
    GAT_HEADS_L2 = 2
    DROPOUT = 0.3
    LR = 1e-3
    WEIGHT_DECAY = 1e-4
    BATCH_SIZE = 8
    EPOCHS = 30
    PATIENCE = 5               # early stopping on val macro-F1
    VAL_SIZE = 0.10
    TEST_SIZE = 0.10

    SYMPTOMS = [
        "Feeling-bad-about-yourself-or-that-you-are-a-failure-or-have-let-yourself-or-your-family-down",
        "Feeling-down-depressed-or-hopeless",
        "Feeling-tired-or-having-little-energy",
        "Little-interest-or-pleasure-in-doing",
        "Moving-or-speaking-so-slowly-that-other-people-could-have-noticed-Or-the-opposite-being-so-fidgety-or-restless-that-you-have-been-moving-around-a-lot-more-than-usual",
        "Poor-appetite-or-overeating",
        "Thoughts-that-you-would-be-better-off-dead-or-of-hurting-yourself-in-some-way",
        "Trouble-concentrating-on-things-such-as-reading-the-newspaper-or-watching-television",
        "Trouble-falling-or-staying-asleep-or-sleeping-too-much",
    ]
    SYMPTOM_SHORT = [
        "worthlessness", "depressed_mood", "fatigue", "anhedonia",
        "psychomotor", "appetite", "suicidal_ideation", "concentration", "sleep",
    ]
    N_SYMPTOMS = len(SYMPTOMS)

    OUT_DIR = "/kaggle/working"
    CHECKPOINT_NAME = "gat_phase2_final.pt"

os.makedirs(CFG.OUT_DIR, exist_ok=True)


## 3. Load and merge the PRIMATE files

`primate_dataset.json` is a list of 2,003 posts. Each `annotations` entry is a list of
`[symptom_name, "yes"/"no"]` pairs. A handful of entries (6 out of 2,003) have a duplicated or
missing symptom name due to an annotation slip — we handle that defensively by building a
dict per post and defaulting anything missing to `"no"` rather than crashing.

`PRIMATE.json` is the 2024 reannotation. Its `primate_id` field is a **direct index** into
`primate_dataset.json` (verified below by cross-checking the quoted span against the actual
post text). Only entries with a non-empty `quote` list carry a usable character span — that's
41 out of 167.


In [12]:
with open(CFG.PRIMATE_BASE_PATH) as f:
    base_posts = json.load(f)

with open(CFG.PRIMATE_2024_PATH) as f:
    reannotated = json.load(f)

print(f"Base dataset (primate_dataset.json): {len(base_posts)} posts")
print(f"2024 reannotation (PRIMATE.json):    {len(reannotated)} entries")

# Build the 9-dim binary label vector for every post.
def extract_labels(post):
    d = {name.strip(): val for name, val in post["annotations"]}
    return np.array([1.0 if d.get(s, "no") == "yes" else 0.0 for s in CFG.SYMPTOMS], dtype=np.float32)

for post in base_posts:
    post["labels"] = extract_labels(post)

label_matrix = np.stack([p["labels"] for p in base_posts])
print("\nSymptom prevalence (positive / total):")
for name, col in zip(CFG.SYMPTOM_SHORT, label_matrix.T):
    print(f"  {name:20s} {int(col.sum()):4d} / {len(base_posts)}  ({col.mean()*100:.1f}%)")


Base dataset (primate_dataset.json): 2003 posts
2024 reannotation (PRIMATE.json):    167 entries

Symptom prevalence (positive / total):
  worthlessness        1680 / 2003  (83.9%)
  depressed_mood       1664 / 2003  (83.1%)
  fatigue               688 / 2003  (34.3%)
  anhedonia             949 / 2003  (47.4%)
  psychomotor           527 / 2003  (26.3%)
  appetite              194 / 2003  (9.7%)
  suicidal_ideation     743 / 2003  (37.1%)
  concentration         195 / 2003  (9.7%)
  sleep                 374 / 2003  (18.7%)


In [13]:
# Keep only the 2024-reannotated entries that actually have a usable evidence span.
explain_examples = [e for e in reannotated if len(e.get("quote", [])) > 0]
print(f"Usable explainability examples (non-empty quote): {len(explain_examples)}")

# Sanity check: primate_id should index directly into base_posts, and the quoted
# characters should be a real, legible substring of that post's text.
sample = explain_examples[0]
post = base_posts[sample["primate_id"]]
s, e = sample["quote"][0]
print(f"\nprimate_id {sample['primate_id']} -> quoted span [{s}:{e}]:")
print(repr(post["post_text"][s:e]))
print("\n(If that reads as a coherent phrase about interest/pleasure/hobbies, the indexing is correct.)")

explain_ids = {e["primate_id"] for e in explain_examples}


Usable explainability examples (non-empty quote): 41

primate_id 1394 -> quoted span [1537:1710]:
"Hobbies, but I don't really have any of those anymore? I've been depressed for so long, I've lost interest in the ones I used to have as a kid, and never developed new ones."

(If that reads as a coherent phrase about interest/pleasure/hobbies, the indexing is correct.)


## 4. Train / val / test split

We hold the 41 explainability posts **out of every split**. They're never used for gradient
updates or for model selection — they exist purely so we can eyeball attention quality after
training on genuinely unseen-in-that-role examples. Everything else is split 80/10/10.


In [14]:
trainable_posts = [p for i, p in enumerate(base_posts) if i not in explain_ids]
print(f"Posts available for train/val/test: {len(trainable_posts)}  "
      f"(held out {len(explain_ids)} for explainability-only)")

train_posts, temp_posts = train_test_split(
    trainable_posts, test_size=(CFG.VAL_SIZE + CFG.TEST_SIZE), random_state=SEED
)
val_posts, test_posts = train_test_split(
    temp_posts, test_size=CFG.TEST_SIZE / (CFG.VAL_SIZE + CFG.TEST_SIZE), random_state=SEED
)
print(f"Train: {len(train_posts)}  Val: {len(val_posts)}  Test: {len(test_posts)}")


Posts available for train/val/test: 1962  (held out 41 for explainability-only)
Train: 1569  Val: 196  Test: 197


## 5. Load the frozen Phase-1 backbone

This is the exact same Mental-RoBERTa checkpoint Phase 1 saved — fine-tuned on Dreaddit then
GoEmotions. Every parameter gets `requires_grad = False`: it becomes a fixed function from text
to embeddings, nothing more, for the rest of this notebook.


In [15]:
!pip install -q sentencepiece

In [10]:
import os
print(os.listdir(checkpoint_path))

['mentalroberta_dreaddit', '__results__.html', 'mentalroberta_phase1_final', '__huggingface_repos__.json', 'dreaddit_ckpt', '__notebook__.ipynb', '__output__.json', 'goemotions_ckpt', 'custom.css']


In [16]:
checkpoint_path = CFG.PHASE1_CHECKPOINT if os.path.exists(CFG.PHASE1_CHECKPOINT) else CFG.FALLBACK_BASE_MODEL
if checkpoint_path == CFG.FALLBACK_BASE_MODEL:
    print(f"WARNING: Phase-1 checkpoint not found at {CFG.PHASE1_CHECKPOINT}. "
          f"Falling back to '{CFG.FALLBACK_BASE_MODEL}' so the rest of this notebook is runnable, "
          f"but results from this fallback are NOT the real Phase-2 model — re-run with the real "
          f"checkpoint before trusting any numbers below.")
else:
    print(f"Loaded Phase-1 checkpoint from {checkpoint_path}")

from transformers import RobertaTokenizer
tokenizer = RobertaTokenizer.from_pretrained(checkpoint_path)
backbone = AutoModel.from_pretrained(checkpoint_path).to(DEVICE)
backbone.eval()
for p in backbone.parameters():
    p.requires_grad = False

EMB_DIM = backbone.config.hidden_size
print("Backbone hidden size:", EMB_DIM)


Loaded Phase-1 checkpoint from /kaggle/input/notebooks/shashwatkashyap12221/phase-1/mentalroberta_phase1_final


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: /kaggle/input/notebooks/shashwatkashyap12221/phase-1/mentalroberta_phase1_final
Key                        | Status     | 
---------------------------+------------+-
classifier.dense.weight    | UNEXPECTED | 
classifier.out_proj.weight | UNEXPECTED | 
classifier.out_proj.bias   | UNEXPECTED | 
classifier.dense.bias      | UNEXPECTED | 
pooler.dense.weight        | MISSING    | 
pooler.dense.bias          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Backbone hidden size: 768


## 6. spaCy dependency graph + subword-to-word alignment

For every post we need two things per token:
1. **A node feature** — a single embedding vector representing that word.
2. **A graph edge** — which other word it's grammatically attached to.

RoBERTa tokenizes into *subwords* ("hopeless" -> "hope", "less"), while spaCy parses into
*words*. To build one node per spaCy token, we average the embeddings of every RoBERTa
subword whose character span falls inside that spaCy token's character span — using the
tokenizer's `offset_mapping` to line the two up exactly.

Dependency edges from spaCy are directed (child -> head). We add the reverse direction too
(so information can flow both ways in message passing) plus self-loops on every node, which is
standard practice for GAT/GCN-style models and keeps a token's own features in the mix after
neighbourhood aggregation.


In [18]:
nlp = spacy.load("en_core_web_sm")

@torch.no_grad()
def build_graph(text, label=None, meta=None):
    
    text = text[:20000]  # guard against pathologically long posts before spaCy even sees them

    doc = nlp(text)
    spacy_tokens = [t for t in doc if not t.is_space]
    if len(spacy_tokens) < 2:
        return None

    enc = tokenizer(
        text,
        return_offsets_mapping=True,
        truncation=True,
        max_length=CFG.MAX_TOKENS,
        return_tensors="pt",
    )
    offsets = enc["offset_mapping"][0].tolist()
    input_ids = enc["input_ids"].to(DEVICE)
    attention_mask = enc["attention_mask"].to(DEVICE)

    out = backbone(input_ids=input_ids, attention_mask=attention_mask)
    subword_embs = out.last_hidden_state[0]  # [n_subwords, hidden]

    # Map each spaCy token -> list of subword indices whose span overlaps it.
    node_feats = []
    kept_tokens = []
    for tok in spacy_tokens:
        tok_start, tok_end = tok.idx, tok.idx + len(tok.text)
        sub_idxs = [
            i for i, (s, e) in enumerate(offsets)
            if not (s == 0 and e == 0) and s < tok_end and e > tok_start
        ]
        if not sub_idxs:
            continue  # token fell entirely outside the truncated window
        node_feats.append(subword_embs[sub_idxs].mean(dim=0))
        kept_tokens.append(tok)

    if len(kept_tokens) < 2:
        return None

    x = torch.stack(node_feats).cpu()

    # Map spaCy token -> its position in kept_tokens (some tokens got dropped above).
    idx_of = {tok.i: pos for pos, tok in enumerate(kept_tokens)}

    edges = []
    for tok in kept_tokens:
        if tok.head.i == tok.i:
            continue  # spaCy marks the sentence root as its own head; self-loop added below anyway
        if tok.head.i in idx_of and tok.i in idx_of:
            a, b = idx_of[tok.i], idx_of[tok.head.i]
            edges.append((a, b))
            edges.append((b, a))  # bidirectional
    for pos in range(len(kept_tokens)):
        edges.append((pos, pos))  # self-loop

    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

    data = Data(x=x, edge_index=edge_index)
    if label is not None:
        data.y = torch.tensor(label, dtype=torch.float32).unsqueeze(0)
    if meta is not None:
        # char offsets per kept node, needed later to map attention back to text
        data.token_spans = [(t.idx, t.idx + len(t.text)) for t in kept_tokens]
        data.primate_id = meta
    return data


## 7. Build the graph datasets

This is the slow cell — every post makes one forward pass through the frozen backbone. On a T4, expect roughly 1-2 posts/second, so ~15-25 minutes for the ~1,800 trainable posts.

In [19]:
def build_split(posts, desc):
    graphs = []
    skipped = 0
    for i, post in enumerate(posts):
        g = build_graph(post["post_text"], label=post["labels"])
        if g is None:
            skipped += 1
            continue
        graphs.append(g)
        if (i + 1) % 200 == 0:
            print(f"  [{desc}] {i+1}/{len(posts)} processed")
    print(f"[{desc}] done: {len(graphs)} graphs built, {skipped} posts skipped (too short/empty)")
    return graphs

train_graphs = build_split(train_posts, "train")
val_graphs = build_split(val_posts, "val")
test_graphs = build_split(test_posts, "test")

train_loader = DataLoader(train_graphs, batch_size=CFG.BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_graphs, batch_size=CFG.BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_graphs, batch_size=CFG.BATCH_SIZE, shuffle=False)


  [train] 200/1569 processed
  [train] 400/1569 processed
  [train] 600/1569 processed
  [train] 800/1569 processed
  [train] 1000/1569 processed
  [train] 1200/1569 processed
  [train] 1400/1569 processed
[train] done: 1569 graphs built, 0 posts skipped (too short/empty)
[val] done: 196 graphs built, 0 posts skipped (too short/empty)
[test] done: 197 graphs built, 0 posts skipped (too short/empty)


## 8. Class imbalance — `pos_weight`

Symptom prevalence ranges from ~10% (poor appetite, concentration) to ~84% (feeling bad about
yourself), from the counts printed in section 3. Left alone, BCE loss will happily predict "no"
for the rare symptoms and still look like it's doing fine. `pos_weight` in
`BCEWithLogitsLoss` up-weights the positive class per symptom to counter that, computed straight
from the training split so it doesn't leak information from val/test.


In [20]:
train_labels = np.stack([g.y.numpy().squeeze(0) for g in train_graphs])
pos_counts = train_labels.sum(axis=0)
neg_counts = len(train_labels) - pos_counts
pos_weight = torch.tensor(neg_counts / np.clip(pos_counts, 1, None), dtype=torch.float32).to(DEVICE)

print("pos_weight per symptom:")
for name, w in zip(CFG.SYMPTOM_SHORT, pos_weight.tolist()):
    print(f"  {name:20s} {w:.2f}")


pos_weight per symptom:
  worthlessness        0.20
  depressed_mood       0.20
  fatigue              1.93
  anhedonia            1.15
  psychomotor          2.80
  appetite             9.39
  suicidal_ideation    1.72
  concentration        9.60
  sleep                4.45


## 9. The GAT model

Two `GATConv` layers:
- **Layer 1** — multi-head attention (4 heads), concatenated, with a ReLU and dropout. Multiple
  heads at this stage let the model learn a few different notions of "what's relevant" (e.g. one
  head might learn to attend to negations, another to symptom-adjacent adjectives).
- **Layer 2** — fewer heads (2), averaged rather than concatenated, to compress back down before
  pooling.
- **Pooling** — `global_mean_pool` collapses all node embeddings in a graph into one
  graph-level vector. Mean (not sum) so post length doesn't bias the magnitude of the pooled
  vector.
- **Classifier head** — a single linear layer to 9 logits, one per PHQ-9 symptom. Sigmoid is
  applied inside the loss (`BCEWithLogitsLoss`) and at inference time, not in the model itself.

`return_attention_weights=True` on both layers lets us pull out attention coefficients later for
the explainability check in section 12 — the forward pass optionally returns them without
changing training behaviour.


In [21]:
class SymptomGAT(nn.Module):
    def __init__(self, in_dim, hidden_dim, n_classes, heads1=4, heads2=2, dropout=0.3):
        super().__init__()
        self.gat1 = GATConv(in_dim, hidden_dim, heads=heads1, dropout=dropout, concat=True)
        self.gat2 = GATConv(hidden_dim * heads1, hidden_dim, heads=heads2, dropout=dropout, concat=False)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim, n_classes)

    def forward(self, x, edge_index, batch, return_attention=False):
        x1, att1 = self.gat1(x, edge_index, return_attention_weights=True)
        x1 = F.elu(x1)
        x1 = self.dropout(x1)

        x2, att2 = self.gat2(x1, edge_index, return_attention_weights=True)
        x2 = F.elu(x2)

        graph_emb = global_mean_pool(x2, batch)
        logits = self.classifier(graph_emb)

        if return_attention:
            return logits, (att1, att2)
        return logits

model = SymptomGAT(
    in_dim=EMB_DIM,
    hidden_dim=CFG.HIDDEN_DIM,
    n_classes=CFG.N_SYMPTOMS,
    heads1=CFG.GAT_HEADS_L1,
    heads2=CFG.GAT_HEADS_L2,
    dropout=CFG.DROPOUT,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable GAT parameters: {n_params:,}")


Trainable GAT parameters: 1,317,385


## 10. Training loop

Only `model.parameters()` (the GAT) are passed to the optimizer — the backbone was frozen in section 5 and never appears here. We track validation macro-F1 for early stopping and checkpoint selection, since macro-F1 treats every symptom equally regardless of how rare it is.

In [22]:
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)


def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []

    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for batch in loader:
            batch = batch.to(DEVICE)
            if train:
                optimizer.zero_grad()
            logits = model(batch.x, batch.edge_index, batch.batch)
            loss = criterion(logits, batch.y)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * batch.num_graphs

            probs = torch.sigmoid(logits).detach().cpu().numpy()
            all_preds.append(probs >= 0.5)
            all_labels.append(batch.y.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return total_loss / len(loader.dataset), macro_f1


best_val_f1 = -1.0
patience_left = CFG.PATIENCE
history = []

for epoch in range(1, CFG.EPOCHS + 1):
    train_loss, train_f1 = run_epoch(train_loader, train=True)
    val_loss, val_f1 = run_epoch(val_loader, train=False)
    history.append((epoch, train_loss, train_f1, val_loss, val_f1))
    print(f"Epoch {epoch:2d} | train loss {train_loss:.4f} f1 {train_f1:.4f} "
          f"| val loss {val_loss:.4f} f1 {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_left = CFG.PATIENCE
        torch.save(model.state_dict(), os.path.join(CFG.OUT_DIR, "gat_phase2_best.pt"))
        print(f"  -> new best val macro-F1 ({val_f1:.4f}), checkpoint saved")
    else:
        patience_left -= 1
        if patience_left == 0:
            print(f"No val improvement for {CFG.PATIENCE} epochs, stopping early.")
            break

model.load_state_dict(torch.load(os.path.join(CFG.OUT_DIR, "gat_phase2_best.pt")))
print(f"\nLoaded best checkpoint (val macro-F1 = {best_val_f1:.4f}) for final evaluation.")


Epoch  1 | train loss 0.8658 f1 0.4375 | val loss 0.8514 f1 0.3782
  -> new best val macro-F1 (0.3782), checkpoint saved
Epoch  2 | train loss 0.8332 f1 0.4688 | val loss 0.8494 f1 0.4222
  -> new best val macro-F1 (0.4222), checkpoint saved
Epoch  3 | train loss 0.8153 f1 0.4738 | val loss 0.8419 f1 0.5021
  -> new best val macro-F1 (0.5021), checkpoint saved
Epoch  4 | train loss 0.8026 f1 0.4858 | val loss 0.8765 f1 0.3749
Epoch  5 | train loss 0.7846 f1 0.4966 | val loss 0.9549 f1 0.4130
Epoch  6 | train loss 0.7715 f1 0.5140 | val loss 0.8875 f1 0.5340
  -> new best val macro-F1 (0.5340), checkpoint saved
Epoch  7 | train loss 0.7602 f1 0.5167 | val loss 0.8456 f1 0.5008
Epoch  8 | train loss 0.7433 f1 0.5280 | val loss 0.8825 f1 0.4822
Epoch  9 | train loss 0.7391 f1 0.5363 | val loss 0.9852 f1 0.3560
Epoch 10 | train loss 0.7335 f1 0.5348 | val loss 0.9314 f1 0.5149
Epoch 11 | train loss 0.7220 f1 0.5471 | val loss 0.8926 f1 0.5270
No val improvement for 5 epochs, stopping early

## 11. Test-set evaluation

In [23]:
test_loss, test_f1 = run_epoch(test_loader, train=False)
print(f"Test loss: {test_loss:.4f}   Test macro-F1: {test_f1:.4f}")

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(DEVICE)
        logits = model(batch.x, batch.edge_index, batch.batch)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_preds.append(probs >= 0.5)
        all_labels.append(batch.y.cpu().numpy())

all_preds = np.concatenate(all_preds)
all_labels = np.concatenate(all_labels)

print("\nPer-symptom test report:\n")
print(classification_report(all_labels, all_preds, target_names=CFG.SYMPTOM_SHORT, zero_division=0))


Test loss: 0.8428   Test macro-F1: 0.5234

Per-symptom test report:

                   precision    recall  f1-score   support

    worthlessness       0.88      0.83      0.86       167
   depressed_mood       0.89      0.88      0.88       167
          fatigue       0.37      0.83      0.51        66
        anhedonia       0.50      0.93      0.65        97
      psychomotor       0.55      0.49      0.52        49
         appetite       0.14      0.47      0.21        17
suicidal_ideation       0.59      0.73      0.65        75
    concentration       0.14      0.57      0.22        21
            sleep       0.23      0.18      0.20        28

        micro avg       0.56      0.78      0.65       687
        macro avg       0.48      0.66      0.52       687
     weighted avg       0.66      0.78      0.69       687
      samples avg       0.57      0.78      0.62       687



## 12. Explainability check — the 41 anhedonia spans

This is validation, not training: none of these 41 posts were in `train_graphs`. For each one
we:
1. Run it through the trained model, keeping attention weights from the second GAT layer.
2. Turn per-edge attention into a single **importance score per node** (sum the attention on
   every edge pointing into that node, averaged across attention heads).
3. Take the top 20% highest-importance tokens and check what fraction of their character spans
   overlap the human-annotated `quote` span for anhedonia.

A high overlap suggests the model is, at least for this symptom, attending to language a human
would also point to as evidence — which is the whole point of putting a GAT in this pipeline
instead of a plain classifier. A low overlap doesn't necessarily mean the model is "wrong" (PHQ-9
symptoms are graph-level here, so nothing forces node-level attention to align with any one
symptom's evidence) — but consistently low overlap across most of the 41 examples would be a
signal worth investigating before trusting the explainability layer downstream.


In [25]:
def node_importance_from_attention(edge_index, alpha, num_nodes):
    alpha = alpha.mean(dim=1)  # average across heads -> [num_edges]
    importance = torch.zeros(num_nodes)
    dst = edge_index[1]
    importance.index_add_(0, dst.cpu(), alpha.cpu())
    return importance


def char_overlap(top_spans, quote_spans):
    covered = 0
    total = sum(e - s for s, e in quote_spans)
    for qs, qe in quote_spans:
        for ts, te in top_spans:
            covered += max(0, min(qe, te) - max(qs, ts))
    return covered / total if total > 0 else 0.0


overlaps = []
model.eval()
with torch.no_grad():
    for ex in explain_examples:
        post = base_posts[ex["primate_id"]]
        g = build_graph(post["post_text"], label=post["labels"], meta=ex["primate_id"])
        if g is None:
            continue
        g = g.to(DEVICE)
        batch_vec = torch.zeros(g.x.size(0), dtype=torch.long, device=DEVICE)
        logits, (att1, att2) = model(g.x, g.edge_index, batch_vec, return_attention=True)

        edge_index2, alpha2 = att2
        importance = node_importance_from_attention(edge_index2, alpha2, g.x.size(0))

        k = max(1, int(0.2 * len(importance)))
        top_idx = torch.topk(importance, k).indices.tolist()
        top_spans = [g.token_spans[i] for i in top_idx]

        quote_spans = [tuple(s) for s in ex["quote"]]
        overlap = char_overlap(top_spans, quote_spans)
        overlaps.append(overlap)

overlaps = np.array(overlaps)
print(f"Evaluated {len(overlaps)} of {len(explain_examples)} explainability examples "
      f"(some may be skipped if truncation drops the evidence span entirely).")
print(f"Mean character overlap between top-20% attended tokens and human-annotated span: "
      f"{overlaps.mean()*100:.1f}%")
print(f"Examples with >50% overlap: {(overlaps > 0.5).sum()} / {len(overlaps)}")
print(f"Examples with 0% overlap:   {(overlaps == 0).sum()} / {len(overlaps)}")


Evaluated 41 of 41 explainability examples (some may be skipped if truncation drops the evidence span entirely).
Mean character overlap between top-20% attended tokens and human-annotated span: 15.9%
Examples with >50% overlap: 4 / 41
Examples with 0% overlap:   18 / 41


In [27]:
# Random-baseline comparison: if we picked 20% of a post's tokens completely at
# random instead of using GAT attention, how much would we "overlap" the
# ground-truth quote just by chance? This tells us whether the 15.9% figure
# above actually reflects the model paying attention to the right words, or
# whether it's within the range you'd get by guessing.

N_RANDOM_TRIALS = 20  # repeat per example so one lucky/unlucky draw doesn't skew it

random_overlaps = []
for ex in explain_examples:
    post = base_posts[ex["primate_id"]]
    g = build_graph(post["post_text"], label=post["labels"], meta=ex["primate_id"])
    if g is None:
        continue

    quote_spans = [tuple(s) for s in ex["quote"]]
    n_tokens = len(g.token_spans)
    k = max(1, int(0.2 * n_tokens))  # same top-20% fraction used for attention

    trial_overlaps = []
    for _ in range(N_RANDOM_TRIALS):
        random_idx = random.sample(range(n_tokens), k)
        random_spans = [g.token_spans[i] for i in random_idx]
        trial_overlaps.append(char_overlap(random_spans, quote_spans))

    random_overlaps.append(np.mean(trial_overlaps))

random_overlaps = np.array(random_overlaps)
print(f"Evaluated {len(random_overlaps)} of {len(explain_examples)} examples "
      f"({N_RANDOM_TRIALS} random draws each, averaged).")
print(f"Mean overlap from RANDOM token selection: {random_overlaps.mean()*100:.1f}%")
print(f"Mean overlap from GAT ATTENTION (from previous cell): {overlaps.mean()*100:.1f}%")
print(f"\nDifference (attention - random): {(overlaps.mean() - random_overlaps.mean())*100:+.1f} percentage points")

if overlaps.mean() > random_overlaps.mean():
    print("Attention beats random selection — some real signal, even if modest.")
else:
    print("Attention does NOT clearly beat random selection on this check — "
          "worth treating current word-level explainability claims cautiously.")

Evaluated 41 of 41 examples (20 random draws each, averaged).
Mean overlap from RANDOM token selection: 14.2%
Mean overlap from GAT ATTENTION (from previous cell): 15.9%

Difference (attention - random): +1.7 percentage points
Attention beats random selection — some real signal, even if modest.


## 13. Save the final checkpoint

Save this as a Kaggle **Output** (or explicitly as a Dataset) before the session ends — `/kaggle/working/` is wiped between sessions unless you do this.

In [26]:
final_path = os.path.join(CFG.OUT_DIR, CFG.CHECKPOINT_NAME)
torch.save({
    "model_state_dict": model.state_dict(),
    "config": {
        "hidden_dim": CFG.HIDDEN_DIM,
        "heads1": CFG.GAT_HEADS_L1,
        "heads2": CFG.GAT_HEADS_L2,
        "dropout": CFG.DROPOUT,
        "in_dim": EMB_DIM,
        "n_classes": CFG.N_SYMPTOMS,
        "symptoms": CFG.SYMPTOMS,
    },
    "test_macro_f1": test_f1,
    "explainability_mean_overlap": float(overlaps.mean()) if len(overlaps) else None,
}, final_path)

print(f"Saved: {final_path}")
print(os.path.exists(final_path))


Saved: /kaggle/working/gat_phase2_final.pt
True


## 14. Summary — what this notebook produced, and what still needs the frozen backbone

- **Training signal:** graph-level 9-way PHQ-9 classification on all trainable posts from
  `primate_dataset.json`, with the 41 anhedonia-span posts held out.
- **Model:** 2-layer GAT on top of the frozen Phase-1 Mental-RoBERTa embeddings + spaCy
  dependency graphs, `pos_weight`-adjusted BCE loss for the class imbalance.
- **Result:** test macro-F1 printed in section 11 — check it against the per-symptom breakdown,
  since macro-F1 alone can hide a model that's only doing well on the common symptoms
  (worthlessness, depressed mood) and poorly on the rare ones (appetite, concentration).
- **Explainability:** the top-20%-attention / quote-span overlap in section 12 is a first read on
  whether the GAT's attention is clinically sensible — not a formal metric, and worth a manual
  look at a few high- and low-overlap examples before drawing conclusions.
- **Checkpoint:** `gat_phase2_final.pt`, ready to be loaded (with the Phase-1 backbone) as the
  frozen-no-more starting point for **Phase 3** — end-to-end training of all three classifier
  heads plus the GAT plus the transformer, jointly, on DepressionEmo + DAIC-WOZ.


